# Deep Learning and GenAI Libraries: PyTorch, Hugging Face and the LLM Toolchain

Notebook for **Chapter 21** of the *python* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/python/ch21.html

Tip: change a number, predict the output, then run the cell again.

## 21.1 What These Libraries Are, When to Use Them, and the Mental Model

In [ ]:
import importlib.util
import importlib.metadata as md

def report(pairs):
    for import_name, pip_name in pairs:
        if importlib.util.find_spec(import_name) is None:
            print(f"{import_name:14s} not installed")
        else:
            try:
                print(f"{import_name:14s} {md.version(pip_name)}")
            except md.PackageNotFoundError:
                print(f"{import_name:14s} installed (no metadata)")

report([("torch", "torch"), ("transformers", "transformers"), ("faiss", "faiss-cpu"),
        ("openai", "openai"), ("anthropic", "anthropic"),
        ("numpy", "numpy"), ("sklearn", "scikit-learn")])

torch          not installed
transformers   not installed
faiss          not installed
openai         not installed
anthropic      not installed
numpy          2.5.3
sklearn        1.9.1


## 21.2 PyTorch Part 1: Tensors and Autograd

In [ ]:
import numpy as np

x = np.arange(24, dtype=np.float32).reshape(2, 3, 4)   # (batch, tokens, features)
print("shape", x.shape, "dtype", x.dtype, "bytes", x.nbytes)

flat = x.reshape(6, 4)            # merge batch and tokens
first_ticket = x[0]               # a slice
copy = x[0].copy()
swapped = x.transpose(0, 2, 1)    # (batch, features, tokens)

print("flat shares memory:", np.shares_memory(x, flat))
print("slice shares memory:", np.shares_memory(x, first_ticket))
print("copy shares memory:", np.shares_memory(x, copy))
print("transposed shape:", swapped.shape, "contiguous:", swapped.flags["C_CONTIGUOUS"])

first_ticket[0, 0] = 99.0          # write through the view
print("x[0,0,0] is now", x[0, 0, 0], "and copy[0,0] is still", copy[0, 0])

shape (2, 3, 4) dtype float32 bytes 96
flat shares memory: True
slice shares memory: True
copy shares memory: False
transposed shape: (2, 4, 3) contiguous: False
x[0,0,0] is now 99.0 and copy[0,0] is still 0.0


In [ ]:
import numpy as np

z = np.array([[2.0, 1.0, 0.1, -1.0],
              [1000.0, 999.0, 998.0, 0.0],
              [0.0, 0.0, 0.0, 0.0]], dtype=np.float64)

def softmax_naive(z):
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def softmax_stable(z):
    z = z - z.max(axis=1, keepdims=True)      # (3,4) - (3,1) broadcasts
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

with np.errstate(over="ignore", invalid="ignore"):
    print("naive row 1:", softmax_naive(z)[1])
print("stable row 1:", np.round(softmax_stable(z)[1], 4))
p = softmax_stable(z)
print("rows sum to", p.sum(axis=1))
print("argmax per row:", p.argmax(axis=1))
print("uniform row:", p[2])

naive row 1: [nan nan nan  0.]
stable row 1: [0.6652 0.2447 0.09   0.    ]
rows sum to [1. 1. 1.]
argmax per row: [0 0 0]
uniform row: [0.25 0.25 0.25 0.25]


In [ ]:
class Value:
    def __init__(self, data, parents=()):
        self.data = float(data)
        self.grad = 0.0
        self._parents = parents
        self._backward = lambda: None

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other))
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __pow__(self, k):
        out = Value(self.data ** k, (self,))
        def _backward():
            self.grad += k * self.data ** (k - 1) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        order, seen = [], set()
        def visit(v):
            if id(v) not in seen:
                seen.add(id(v))
                for p in v._parents:
                    visit(p)
                order.append(v)
        visit(self)
        self.grad = 1.0
        for v in reversed(order):
            v._backward()

def f(xv, yv):
    x, y = Value(xv), Value(yv)
    z = x * y + x ** 2
    return x, y, z

x, y, z = f(2.0, -3.0)
z.backward()
print("z =", z.data, " dz/dx =", x.grad, " dz/dy =", y.grad)

h = 1e-6
num_dx = (f(2.0 + h, -3.0)[2].data - f(2.0 - h, -3.0)[2].data) / (2 * h)
num_dy = (f(2.0, -3.0 + h)[2].data - f(2.0, -3.0 - h)[2].data) / (2 * h)
print("numeric dx, dy:", round(num_dx, 4), round(num_dy, 4))

z.backward()                      # call again without zeroing
print("after second backward, x.grad =", x.grad)

z = -2.0  dz/dx = 1.0  dz/dy = 2.0
numeric dx, dy: 1.0 2.0
after second backward, x.grad = 3.0


In [ ]:
import numpy as np

rng = np.random.default_rng(0)
N, D = 50, 3
X = rng.normal(size=(N, D))
true_w = np.array([2.0, -1.0, 0.5])
y = X @ true_w + 0.3 + rng.normal(scale=0.1, size=N)

def loss_fn(w, b):
    err = X @ w + b - y
    return np.mean(err ** 2)

def analytic_grads(w, b):
    err = X @ w + b - y
    return 2 / N * X.T @ err, 2 / N * err.sum()

def numeric_grads(w, b, h=1e-6):
    gw = np.zeros_like(w)
    for i in range(len(w)):
        e = np.zeros_like(w); e[i] = h
        gw[i] = (loss_fn(w + e, b) - loss_fn(w - e, b)) / (2 * h)
    gb = (loss_fn(w, b + h) - loss_fn(w, b - h)) / (2 * h)
    return gw, gb

w0, b0 = np.zeros(D), 0.0
gw, gb = analytic_grads(w0, b0)
nw, nb = numeric_grads(w0, b0)
print("analytic dw:", np.round(gw, 4), "db:", round(gb, 4))
print("numeric  dw:", np.round(nw, 4), "db:", round(nb, 4))
print("match:", np.allclose(gw, nw, atol=1e-6) and abs(gb - nb) < 1e-6)

analytic dw: [-4.941   1.0721 -1.4741] db: 0.0996
numeric  dw: [-4.941   1.0721 -1.4741] db: 0.0996
match: True


## 21.3 PyTorch Part 2: nn.Module, Losses, Optimisers and Schedulers

In [ ]:
import numpy as np

class Parameter:
    def __init__(self, data):
        self.data = data
        self.grad = np.zeros_like(data)

class Module:
    def __init__(self):
        object.__setattr__(self, "_params", {})
        object.__setattr__(self, "_mods", {})
        object.__setattr__(self, "training", True)
    def __setattr__(self, name, value):
        if isinstance(value, Parameter):
            self._params[name] = value
        elif isinstance(value, Module):
            self._mods[name] = value
        object.__setattr__(self, name, value)
    def named_parameters(self, prefix=""):
        for k, p in self._params.items():
            yield prefix + k, p
        for k, m in self._mods.items():
            yield from m.named_parameters(prefix + k + ".")
    def parameters(self):
        return [p for _, p in self.named_parameters()]
    def zero_grad(self):
        for p in self.parameters():
            p.grad[...] = 0.0
    def train(self, mode=True):
        object.__setattr__(self, "training", mode)
        for m in self._mods.values():
            m.train(mode)
        return self
    def eval(self):
        return self.train(False)

class Linear(Module):
    def __init__(self, n_in, n_out, rng):
        super().__init__()
        self.weight = Parameter(rng.normal(scale=np.sqrt(2 / n_in), size=(n_in, n_out)))
        self.bias = Parameter(np.zeros(n_out))
    def forward(self, x):
        self.x = x
        return x @ self.weight.data + self.bias.data
    def backward(self, g):
        self.weight.grad += self.x.T @ g
        self.bias.grad += g.sum(axis=0)
        return g @ self.weight.data.T

class ReLU(Module):
    def forward(self, x):
        self.mask = x > 0
        return x * self.mask
    def backward(self, g):
        return g * self.mask

class Sequential(Module):
    def __init__(self, *layers):
        super().__init__()
        for i, layer in enumerate(layers):
            setattr(self, str(i), layer)
        self.layers = layers
    def forward(self, x):
        for layer in self.layers:
            x = layer.forward(x)
        return x
    def backward(self, g):
        for layer in reversed(self.layers):
            g = layer.backward(g)
        return g

rng = np.random.default_rng(1)
model = Sequential(Linear(2, 16, rng), ReLU(), Linear(16, 2, rng))
for name, p in model.named_parameters():
    print(f"{name:10s} {p.data.shape}")
print("total parameters:", sum(p.data.size for p in model.parameters()))

x = rng.normal(size=(5, 2))
out = model.forward(x)
print("output shape:", out.shape)
model.zero_grad()
model.backward(np.ones_like(out))            # gradient of sum(out) w.r.t. out is 1
analytic = model.layers[0].weight.grad[0, 0]

w = model.layers[0].weight.data
h = 1e-6
w[0, 0] += h; up = model.forward(x).sum()
w[0, 0] -= 2 * h; down = model.forward(x).sum()
w[0, 0] += h
print("analytic vs numeric:", round(analytic, 5), round((up - down) / (2 * h), 5))

0.weight   (2, 16)
0.bias     (16,)
2.weight   (16, 2)
2.bias     (2,)
total parameters: 82
output shape: (5, 2)
analytic vs numeric: 0.97713 0.97713


In [ ]:
import numpy as np

rng = np.random.default_rng(0)

def dropout(x, p, training, rng):
    if not training or p == 0:
        return x
    keep = rng.random(x.shape) >= p
    return x * keep / (1 - p)

x = np.ones((1, 10))
print("train call 1:", dropout(x, 0.5, True, rng))
print("train call 2:", dropout(x, 0.5, True, rng))
print("eval  call 1:", dropout(x, 0.5, False, rng))

big = np.ones((10000, 1))
print("train mean over 10000 values:", round(dropout(big, 0.3, True, rng).mean(), 3))

train call 1: [[2. 0. 0. 0. 2. 2. 2. 2. 2. 2.]]
train call 2: [[2. 0. 2. 0. 2. 0. 2. 2. 0. 0.]]
eval  call 1: [[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]]
train mean over 10000 values: 0.999


In [ ]:
import numpy as np

def cross_entropy_from_logits(z, y):
    z = z - z.max(axis=1, keepdims=True)
    log_probs = z - np.log(np.exp(z).sum(axis=1, keepdims=True))
    return -log_probs[np.arange(len(y)), y].mean()

def naive_cross_entropy(z, y):
    p = np.exp(z) / np.exp(z).sum(axis=1, keepdims=True)
    return -np.log(p[np.arange(len(y)), y]).mean()

z = np.array([[2.0, 0.5, -1.0]])
y = np.array([0])
p = np.exp(z) / np.exp(z).sum()
print("softmax:", np.round(p, 4))
print("loss:", round(cross_entropy_from_logits(z, y), 4))
print("gradient wrt logits:", np.round(p - np.eye(3)[y], 4))

huge = np.array([[1000.0, 0.0, -5.0]])
with np.errstate(all="ignore"):
    print("naive on huge logits:", naive_cross_entropy(huge, y))
print("stable on huge logits:", cross_entropy_from_logits(huge, y) + 0.0)

# binary version with logits: log(1 + exp(-t*z)) written stably
def bce_with_logits(z, t):
    return np.mean(np.maximum(z, 0) - z * t + np.log1p(np.exp(-np.abs(z))))
print("BCE:", round(bce_with_logits(np.array([2.0, -1.0, 0.5]), np.array([1.0, 0.0, 1.0])), 4))

softmax: [[0.7856 0.1753 0.0391]]
loss: 0.2413
gradient wrt logits: [[-0.2144  0.1753  0.0391]]
naive on huge logits: nan
stable on huge logits: 0.0
BCE: 0.3048


In [ ]:
import numpy as np

def f(w):
    return 0.5 * (w[0] ** 2 + 100 * w[1] ** 2)

def grad(w):
    return np.array([w[0], 100 * w[1]])

def run(step_fn, steps=100):
    w = np.array([5.0, 5.0])
    state = {}
    for t in range(1, steps + 1):
        w = step_fn(w, grad(w), state, t)
    return f(w)

def sgd(lr):
    def step(w, g, s, t):
        return w - lr * g
    return step

def momentum(lr, beta=0.9):
    def step(w, g, s, t):
        s["v"] = beta * s.get("v", 0) + g
        return w - lr * s["v"]
    return step

def adam(lr, b1=0.9, b2=0.999, eps=1e-8):
    def step(w, g, s, t):
        s["m"] = b1 * s.get("m", 0) + (1 - b1) * g
        s["v"] = b2 * s.get("v", 0) + (1 - b2) * g ** 2
        mh = s["m"] / (1 - b1 ** t)
        vh = s["v"] / (1 - b2 ** t)
        return w - lr * mh / (np.sqrt(vh) + eps)
    return step

print("start loss:", f(np.array([5.0, 5.0])))
print("SGD lr=0.019      :", round(run(sgd(0.019)), 6))
print("SGD lr=0.021 (too big):", f"{run(sgd(0.021)):.3e}")
print("momentum lr=0.01  :", round(run(momentum(0.01)), 6))
print("Adam lr=0.3       :", round(run(adam(0.3)), 6))

start loss: 1262.5
SGD lr=0.019      : 0.269596
SGD lr=0.021 (too big): 2.374e+11
momentum lr=0.01  : 0.002365
Adam lr=0.3       : 0.023447


In [ ]:
import numpy as np

def lr_at(step, total, warmup, peak, floor=0.0):
    if step < warmup:
        return peak * step / warmup
    prog = (step - warmup) / max(1, total - warmup)
    return floor + 0.5 * (peak - floor) * (1 + np.cos(np.pi * prog))

for s in [0, 50, 100, 300, 550, 1000]:
    print(f"step {s:4d}  lr {lr_at(s, 1000, 100, 3e-4):.2e}")

def clip_grad_norm(grads, max_norm):
    total = np.sqrt(sum((g ** 2).sum() for g in grads))
    scale = min(1.0, max_norm / (total + 1e-6))
    return [g * scale for g in grads], total

grads = [np.array([3.0, 4.0]), np.array([12.0])]      # global norm = sqrt(9+16+144) = 13
clipped, before = clip_grad_norm(grads, 1.0)
after = np.sqrt(sum((g ** 2).sum() for g in clipped))
print("norm before:", round(before, 3), "after:", round(after, 3))
print("direction kept:", np.round(clipped[0] / np.linalg.norm(clipped[0]), 3), "= [0.6, 0.8]")

step    0  lr 0.00e+00
step   50  lr 1.50e-04
step  100  lr 3.00e-04
step  300  lr 2.65e-04
step  550  lr 1.50e-04
step 1000  lr 0.00e+00
norm before: 13.0 after: 1.0
direction kept: [0.6 0.8] = [0.6, 0.8]


## 21.4 PyTorch Part 3: Datasets, DataLoader, the Training Loop and Saving

In [ ]:
import numpy as np

class TicketDataset:
    def __init__(self, seqs, labels):
        self.seqs, self.labels = seqs, labels
    def __len__(self):
        return len(self.seqs)
    def __getitem__(self, i):
        return self.seqs[i], self.labels[i]

def pad_collate(items, pad_id=0):
    seqs, labels = zip(*items)
    longest = max(len(s) for s in seqs)
    ids = np.full((len(seqs), longest), pad_id, dtype=np.int64)
    mask = np.zeros((len(seqs), longest), dtype=np.int64)
    for r, s in enumerate(seqs):
        ids[r, :len(s)] = s
        mask[r, :len(s)] = 1
    return ids, mask, np.array(labels, dtype=np.int64)

class DataLoader:
    def __init__(self, ds, batch_size, shuffle=False, drop_last=False, collate_fn=pad_collate, seed=0):
        self.ds, self.bs, self.shuffle = ds, batch_size, shuffle
        self.drop_last, self.collate, self.rng = drop_last, collate_fn, np.random.default_rng(seed)
    def __len__(self):
        n = len(self.ds)
        return n // self.bs if self.drop_last else -(-n // self.bs)
    def __iter__(self):
        order = self.rng.permutation(len(self.ds)) if self.shuffle else np.arange(len(self.ds))
        for start in range(0, len(order), self.bs):
            idx = order[start:start + self.bs]
            if self.drop_last and len(idx) < self.bs:
                break
            yield self.collate([self.ds[int(i)] for i in idx])

seqs = [[5, 6, 7], [8, 9, 10, 11, 12], [13, 14], [15, 16, 17, 18], [19, 20, 21, 22, 23, 24], [25, 26], [27, 28, 29]]
ds = TicketDataset(seqs, [0, 1, 2, 0, 1, 2, 0])

dl = DataLoader(ds, batch_size=3)
print("batches:", len(dl), "| with drop_last:", len(DataLoader(ds, 3, drop_last=True)))
for ids, mask, y in dl:
    print("ids", ids.shape, "mask", mask.shape, "labels", y)
ids, mask, y = next(iter(dl))
print(ids)
print(mask)
first = [int(y.sum()) for _, _, y in DataLoader(ds, 3, shuffle=True, seed=1)]
print("shuffled label sums per batch:", first)

batches: 3 | with drop_last: 2
ids (3, 5) mask (3, 5) labels [0 1 2]
ids (3, 6) mask (3, 6) labels [0 1 2]
ids (1, 3) mask (1, 3) labels [0]
[[ 5  6  7  0  0]
 [ 8  9 10 11 12]
 [13 14  0  0  0]]
[[1 1 1 0 0]
 [1 1 1 1 1]
 [1 1 0 0 0]]
shuffled label sums per batch: [3, 3, 0]


In [ ]:
import numpy as np, tempfile, os
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=400, noise=0.35, random_state=0)
X = (X - X.mean(0)) / X.std(0)
Xtr, ytr, Xva, yva = X[:60], y[:60], X[60:], y[60:]

class MLP:
    def __init__(self, sizes, rng):
        self.W = [rng.normal(scale=np.sqrt(2 / a), size=(a, b)) for a, b in zip(sizes[:-1], sizes[1:])]
        self.b = [np.zeros(b) for b in sizes[1:]]
    def params(self):
        return self.W + self.b
    def forward(self, x):
        self.acts = [x]
        for i, (W, b) in enumerate(zip(self.W, self.b)):
            x = x @ W + b
            if i < len(self.W) - 1:
                x = np.maximum(x, 0)
            self.acts.append(x)
        return x
    def backward(self, g):
        gW, gb = [None] * len(self.W), [None] * len(self.W)
        for i in reversed(range(len(self.W))):
            gW[i] = self.acts[i].T @ g
            gb[i] = g.sum(0)
            if i > 0:
                g = (g @ self.W[i].T) * (self.acts[i] > 0)
        return gW + gb

class Adam:
    def __init__(self, params, lr=0.01, b1=0.9, b2=0.999, eps=1e-8):
        self.p, self.lr, self.b1, self.b2, self.eps, self.t = params, lr, b1, b2, eps, 0
        self.m = [np.zeros_like(q) for q in params]
        self.v = [np.zeros_like(q) for q in params]
    def step(self, grads):
        self.t += 1
        for q, g, m, v in zip(self.p, grads, self.m, self.v):
            m *= self.b1; m += (1 - self.b1) * g
            v *= self.b2; v += (1 - self.b2) * g * g
            q -= self.lr * (m / (1 - self.b1 ** self.t)) / (np.sqrt(v / (1 - self.b2 ** self.t)) + self.eps)

def ce_loss_grad(logits, y):
    z = logits - logits.max(1, keepdims=True)
    p = np.exp(z); p /= p.sum(1, keepdims=True)
    loss = -np.log(p[np.arange(len(y)), y]).mean()
    p[np.arange(len(y)), y] -= 1
    return loss, p / len(y)

def evaluate(model, X, y):                  # like model.eval() + torch.no_grad()
    logits = model.forward(X)
    loss, _ = ce_loss_grad(logits, y)
    return loss, (logits.argmax(1) == y).mean()

rng = np.random.default_rng(0)
model = MLP([2, 64, 2], rng)
opt = Adam(model.params(), lr=0.01)
best_loss, best_state, best_epoch, bad = 1e9, None, -1, 0

for epoch in range(1, 301):
    order = rng.permutation(len(Xtr))
    total = 0.0
    for i in range(0, len(order), 16):
        idx = order[i:i + 16]
        logits = model.forward(Xtr[idx])              # forward
        loss, g = ce_loss_grad(logits, ytr[idx])      # loss
        grads = model.backward(g)                     # backward
        opt.step(grads)                               # optimiser step
        total += loss * len(idx)
    val_loss, val_acc = evaluate(model, Xva, yva)
    if val_loss < best_loss - 1e-4:
        best_loss, best_epoch, bad = val_loss, epoch, 0
        best_state = [q.copy() for q in model.params()]
    else:
        bad += 1
    if epoch % 20 == 0:
        print(f"epoch {epoch:3d} train {total/len(Xtr):.4f} val {val_loss:.4f} acc {val_acc:.3f}")
    if bad >= 30:
        print(f"early stop at epoch {epoch}; best epoch {best_epoch} (val loss {best_loss:.4f})")
        break

path = os.path.join(tempfile.mkdtemp(), "best.npz")
np.savez(path, *best_state)                                   # like torch.save(state_dict)
fresh = MLP([2, 64, 2], np.random.default_rng(99))            # different random init
loaded = np.load(path)
for q, key in zip(fresh.params(), loaded.files):
    q[...] = loaded[key]                                      # like load_state_dict
for q, s in zip(model.params(), best_state):
    q[...] = s
print("reloaded predictions identical:", np.array_equal(fresh.forward(Xva), model.forward(Xva)))
print("final val accuracy of best model:", round(evaluate(fresh, Xva, yva)[1], 3))

epoch  20 train 0.3745 val 0.3671 acc 0.847
epoch  40 train 0.3097 val 0.3361 acc 0.862
epoch  60 train 0.2553 val 0.3559 acc 0.847
epoch  80 train 0.2272 val 0.3671 acc 0.847
early stop at epoch 83; best epoch 53 (val loss 0.3332)
reloaded predictions identical: True
final val accuracy of best model: 0.865


## 21.5 Devices, Memory and Mixed Precision

In [ ]:
def training_gb(n_params, w_bytes, g_bytes, opt_bytes, trainable_frac=1.0, base_bytes=None):
    """Bytes for weights + grads + optimiser state (activations excluded)."""
    trainable = n_params * trainable_frac
    frozen = n_params - trainable
    base = frozen * (base_bytes if base_bytes is not None else w_bytes)
    total = base + trainable * (w_bytes + g_bytes + opt_bytes)
    return total / 1e9

P = 7e9
rows = {
    "full fine-tune, fp32 + Adam":      training_gb(P, 4, 4, 8),
    "all bf16 (weights, grads, Adam)":   training_gb(P, 2, 2, 4),
    "LoRA 0.5% trainable, bf16 base":    training_gb(P, 4, 4, 8, trainable_frac=0.005, base_bytes=2),
    "inference, bf16":                   P * 2 / 1e9,
    "inference, 4-bit quantised":        P * 0.5 / 1e9,
}
for name, gb in rows.items():
    print(f"{name:34s} {gb:7.1f} GB")

# batch size finder: halve until the estimate fits, like an OOM retry loop
def max_batch(budget_gb, fixed_gb, per_sample_gb, start=256):
    b = start
    while b > 0 and fixed_gb + b * per_sample_gb > budget_gb:
        b //= 2
    return b
print("largest batch that fits 24 GB with 14.5 GB fixed:", max_batch(24, 14.5, 0.35))

full fine-tune, fp32 + Adam          112.0 GB
all bf16 (weights, grads, Adam)       56.0 GB
LoRA 0.5% trainable, bf16 base        14.5 GB
inference, bf16                       14.0 GB
inference, 4-bit quantised             3.5 GB
largest batch that fits 24 GB with 14.5 GB fixed: 16


In [ ]:
import numpy as np

print("float16 max:", np.finfo(np.float16).max, " eps:", np.finfo(np.float16).eps)
print("float32 max:", f"{np.finfo(np.float32).max:.3e}", " eps:", f"{np.finfo(np.float32).eps:.3e}")
print("bfloat16 max (computed): ", f"{(2 - 2**-7) * 2.0**127:.3e}", " eps:", 2**-7)

big = np.float32(70000.0)
print("70000 in float16:", np.float16(big))

# emulate bfloat16 by keeping only the top 16 bits of a float32
def to_bf16(x):
    u = np.asarray(x, dtype=np.float32).view(np.uint32)
    return (u & np.uint32(0xFFFF0000)).view(np.float32)
print("70000 in bfloat16 (truncated):", to_bf16(70000.0))

# small updates vanish when added to a large weight
w16 = np.float16(1.0)
print("1.0 + 0.0004 in float16:", w16 + np.float16(0.0004))
print("1.0 + 0.0004 in float32:", np.float32(1.0) + np.float32(0.0004))

# accumulating in low precision stalls
s16, s32 = np.float16(0.0), np.float32(0.0)
for _ in range(20000):
    s16 = np.float16(s16 + np.float16(0.0001))
    s32 = np.float32(s32 + np.float32(0.0001))
print("sum of 20000 x 0.0001 -> float16:", s16, " float32:", round(float(s32), 4))

float16 max: 6.55e+04  eps: 0.000977
float32 max: 3.403e+38  eps: 1.192e-07
bfloat16 max (computed):  3.390e+38  eps: 0.0078125
70000 in float16: inf
70000 in bfloat16 (truncated): 69632.0
1.0 + 0.0004 in float16: 1.0
1.0 + 0.0004 in float32: 1.0004
sum of 20000 x 0.0001 -> float16: 0.25  float32: 2.0002


In [ ]:
import numpy as np

rng = np.random.default_rng(3)
X = rng.normal(size=(64, 5)); y = rng.normal(size=64)
w = rng.normal(size=5)

def grad(Xb, yb, w):                       # gradient of the MEAN squared error
    return 2 / len(yb) * Xb.T @ (Xb @ w - yb)

full = grad(X, y, w)

acc = np.zeros(5)
for i in range(0, 64, 16):                 # 4 micro-batches of 16
    acc += grad(X[i:i + 16], y[i:i + 16], w) / 4      # divide by the number of micro-batches
print("identical to full batch:", np.allclose(full, acc))

wrong = np.zeros(5)
for i in range(0, 64, 16):
    wrong += grad(X[i:i + 16], y[i:i + 16], w)        # forgot the division
print("without /4 the gradient is", round(float(wrong[0] / full[0]), 3), "times too large")

identical to full batch: True
without /4 the gradient is 4.0 times too large


## 21.6 The Shape-Debugging Guide

Most deep learning bugs are shape bugs, and the most dangerous shape bugs do not raise errors: broadcasting silently produces an answer of the wrong size or meaning. The cure is a habit: **write the shape of every tensor in a comment, print shapes at the first batch, and assert the ones that matter.**

In [ ]:
import numpy as np

def check(x, spec, **dims):
    """Assert a shape against letters, e.g. check(x, 'B T E', B=2)."""
    names = spec.split()
    assert x.ndim == len(names), f"expected {len(names)} dims ({spec}), got shape {x.shape}"
    for n, s in zip(names, x.shape):
        if n in dims:
            assert dims[n] == s, f"dim {n}: expected {dims[n]}, got {s} (shape {x.shape})"
        dims[n] = s
    return dims

rng = np.random.default_rng(0)
B, T, E, H = 2, 5, 8, 2
D = E // H
x = rng.normal(size=(B, T, E)); check(x, "B T E", B=B, T=T, E=E)
Wq, Wk, Wv, Wo = (rng.normal(scale=0.3, size=(E, E)) for _ in range(4))
mask = np.array([[1, 1, 1, 1, 1], [1, 1, 1, 0, 0]])          # second ticket has 2 pad tokens

q = x @ Wq                                                     # (B, T, E)
k = x @ Wk
v = x @ Wv
def split_heads(t):                                            # t.view(B, T, H, D).transpose(1, 2)
    return t.reshape(B, T, H, D).transpose(0, 2, 1, 3)         # (B, H, T, D)
qh, kh, vh = split_heads(q), split_heads(k), split_heads(v)
print("heads      :", qh.shape)

scores = qh @ kh.transpose(0, 1, 3, 2) / np.sqrt(D)            # (B, H, T, T)
print("scores     :", scores.shape)
scores = np.where(mask[:, None, None, :] == 1, scores, -1e9)   # mask: (B,1,1,T) broadcasts over H and rows
scores = scores - scores.max(-1, keepdims=True)
att = np.exp(scores); att /= att.sum(-1, keepdims=True)        # softmax over the LAST dim (keys)
print("attention  :", att.shape, " rows sum to 1:", np.allclose(att.sum(-1), 1))
print("pad weight in ticket 2:", round(float(att[1, :, :, 3:].sum()), 6))

ctx = att @ vh                                                 # (B, H, T, D)
merged = ctx.transpose(0, 2, 1, 3).reshape(B, T, E)            # back to (B, T, E)
out = merged @ Wo
print("context    :", ctx.shape, " merged:", merged.shape, " out:", out.shape)
print(check(out, "B T E", B=B, T=T, E=E))

try:
    check(out.reshape(B * T, E), "B T E", B=B)
except AssertionError as e:
    print("caught:", e)

heads      : (2, 2, 5, 4)
scores     : (2, 2, 5, 5)
attention  : (2, 2, 5, 5)  rows sum to 1: True
pad weight in ticket 2: 0.0
context    : (2, 2, 5, 4)  merged: (2, 5, 8)  out: (2, 5, 8)
{'B': 2, 'T': 5, 'E': 8}
caught: expected 3 dims (B T E), got shape (10, 8)


In [ ]:
import numpy as np

pred = np.array([0.9, 0.2, 0.7, 0.4])            # shape (4,)
target = np.array([[1.0], [0.0], [1.0], [0.0]])  # shape (4, 1), e.g. from y.unsqueeze(1)

diff = pred - target                             # (4,) - (4,1) -> (4,4)
print("diff shape:", diff.shape)
print("wrong MSE :", round(float(np.mean(diff ** 2)), 4))

fixed = pred - target.ravel()                    # both (4,)
print("right MSE :", round(float(np.mean(fixed ** 2)), 4))

diff shape: (4, 4)
wrong MSE : 0.325
right MSE : 0.075


## 21.7 Hugging Face: transformers, tokenizers, datasets and peft

In [ ]:
from collections import Counter

corpus = ["refund the billing charge", "billing charge refunded twice", "cannot reset password",
          "password reset link broken", "charge appears twice on billing page", "reset the password link",
          "refund requested for twice charged order", "how to reset billing password"]

def train_bpe(texts, n_merges):
    words = Counter()
    for t in texts:
        for w in t.lower().split():
            words[tuple(w) + ("</w>",)] += 1
    merges = []
    for _ in range(n_merges):
        pairs = Counter()
        for w, c in words.items():
            for a, b in zip(w, w[1:]):
                pairs[(a, b)] += c
        if not pairs:
            break
        best = max(pairs, key=lambda p: (pairs[p], p))      # most frequent pair, ties by name
        merges.append(best)
        new = Counter()
        for w, c in words.items():
            out, i = [], 0
            while i < len(w):
                if i < len(w) - 1 and (w[i], w[i + 1]) == best:
                    out.append(w[i] + w[i + 1]); i += 2
                else:
                    out.append(w[i]); i += 1
            new[tuple(out)] += c
        words = new
    return merges

class BPETokenizer:
    def __init__(self, merges, corpus):
        self.rank = {m: i for i, m in enumerate(merges)}
        chars = sorted({ch for t in corpus for ch in t.lower() if ch != " "})
        toks = ["<pad>", "<unk>"] + chars + ["</w>"] + ["".join(m) for m in merges]
        self.vocab = {t: i for i, t in enumerate(dict.fromkeys(toks))}
        self.inv = {i: t for t, i in self.vocab.items()}
    def tokenize(self, text):
        out = []
        for w in text.lower().split():
            syms = list(w) + ["</w>"]
            while len(syms) > 1:
                cand = [(self.rank[p], i) for i, p in enumerate(zip(syms, syms[1:])) if p in self.rank]
                if not cand:
                    break
                _, i = min(cand)                              # apply the earliest-learned merge first
                syms[i:i + 2] = [syms[i] + syms[i + 1]]
            out.extend(syms)
        return out
    def __call__(self, texts, padding=False, truncation=False, max_length=None):
        ids = [[self.vocab.get(t, 1) for t in self.tokenize(x)] for x in texts]
        if truncation and max_length:
            ids = [s[:max_length] for s in ids]
        longest = max(len(s) for s in ids) if padding else 0
        mask = [[1] * len(s) + [0] * (longest - len(s)) for s in ids]
        ids = [s + [0] * (longest - len(s)) for s in ids]
        return {"input_ids": ids, "attention_mask": mask}
    def decode(self, ids):
        return "".join(self.inv[i] for i in ids if i != 0).replace("</w>", " ").strip()

merges = train_bpe(corpus, 50)
tok = BPETokenizer(merges, corpus)
print("first merges:", merges[:5])
print("vocab size:", len(tok.vocab))
print("seen    :", tok.tokenize("refund the billing"))
print("unseen  :", tok.tokenize("refunding passwords"))
print("unknown :", tok.tokenize("xyz"))
batch = tok(["reset password", "refund the billing charge twice"], padding=True)
print("shape   :", len(batch["input_ids"]), "x", len(batch["input_ids"][0]))
print("mask    :", batch["attention_mask"])
print("decoded :", tok.decode(batch["input_ids"][0]))
print("truncated:", tok(["refund the billing charge twice"], truncation=True, max_length=4)["input_ids"])

first merges: [('e', '</w>'), ('d', '</w>'), ('r', 'e'), ('o', 'r'), ('l', 'i')]
vocab size: 73
seen    : ['refund</w>', 'the</w>', 'billing</w>']
unseen  : ['refund', 'i', 'n', 'g', '</w>', 'pa', 's', 's', 'wor', 'd', 's</w>']
unknown : ['x', 'y', 'z', '</w>']
shape   : 2 x 5
mask    : [[1, 1, 0, 0, 0], [1, 1, 1, 1, 1]]
decoded : reset password
truncated: [[59, 58, 47, 56]]


In [ ]:
import io
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.DataFrame({
    "text": ["refund the charge", "app crashes on login", "how to export data", "charged twice", "button does nothing",
             "reset my password please", "invoice missing", "error 500 on save", "where is the api key", "ok"],
    "label": ["billing", "bug", "how-to", "billing", "bug", "how-to", "billing", "bug", "how-to", "bug"],
})
print("text dtype:", df["text"].dtype)

def add_tokens(batch):                       # batched map: dict of lists in, dict of lists out
    return {"n_tokens": [len(t.split()) for t in batch["text"]]}

new_cols = add_tokens({"text": df["text"].tolist()})
df = df.assign(**new_cols)                   # returns a NEW frame, like Dataset.map
df = df[df["n_tokens"] >= 3].reset_index(drop=True)      # like Dataset.filter
print("rows kept:", len(df))

train, test = train_test_split(df, test_size=0.2, random_state=0, stratify=None)
print("train/test:", len(train), len(test), "| test rows:", list(test["text"]))

csv_text = df.to_csv(index=False)
seen = 0
for chunk in pd.read_csv(io.StringIO(csv_text), chunksize=4):     # like streaming=True
    seen += len(chunk)
    print("chunk of", len(chunk), "rows; first label:", chunk["label"].iloc[0])
print("streamed rows:", seen)

text dtype: str
rows kept: 7
train/test: 5 2 | test rows: ['where is the api key', 'how to export data']
chunk of 4 rows; first label: billing
chunk of 3 rows; first label: how-to
streamed rows: 7


In [ ]:
import numpy as np

rng = np.random.default_rng(0)
d, r, N = 64, 4, 512
W0 = rng.normal(size=(d, d)) / np.sqrt(d)                       # frozen base weight
delta = rng.normal(size=(d, r)) @ rng.normal(size=(r, d)) / np.sqrt(d * r)   # hidden rank-4 change
X = rng.normal(size=(N, d))
Y = X @ (W0 + delta).T                                          # targets from the "true" fine-tuned layer

alpha = 8
s = alpha / r
A = rng.normal(scale=0.01, size=(r, d))
B = np.zeros((d, r))

def forward_loss():
    E = X @ (W0 + s * B @ A).T - Y
    return (E ** 2).mean(), E

print("loss at start (base model):", round(forward_loss()[0], 4))
mA, vA, mB, vB = np.zeros_like(A), np.zeros_like(A), np.zeros_like(B), np.zeros_like(B)
for t in range(1, 501):
    loss, E = forward_loss()
    G = 2 / (N * d) * E.T @ X                                   # dL/d(effective weight), shape (d, d)
    gB, gA = s * G @ A.T, s * B.T @ G
    for p, g, m, v in ((A, gA, mA, vA), (B, gB, mB, vB)):       # Adam, lr 0.02
        m *= 0.9; m += 0.1 * g
        v *= 0.999; v += 0.001 * g * g
        p -= 0.02 * (m / (1 - 0.9 ** t)) / (np.sqrt(v / (1 - 0.999 ** t)) + 1e-8)

print("loss after 500 steps:", f"{forward_loss()[0]:.2e}")
print("trainable:", A.size + B.size, "of", W0.size, f"= {100 * (A.size + B.size) / W0.size:.1f}%")
W_merged = W0 + s * B @ A                                       # merge_and_unload
unmerged = X @ W0.T + s * (X @ A.T) @ B.T                       # adapter applied on the side
print("merged equals unmerged:", np.allclose(X @ W_merged.T, unmerged))
print("rank of learned update:", np.linalg.matrix_rank(s * B @ A))
print("max error vs true weights:", f"{np.abs(W_merged - (W0 + delta)).max():.1e}")

loss at start (base model): 1.0581
loss after 500 steps: 1.48e-10
trainable: 512 of 4096 = 12.5%
merged equals unmerged: True
rank of learned update: 4
max error vs true weights: 2.6e-05


## 21.8 Embeddings and Vector Search: sentence-transformers, FAISS and Chroma

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline

docs = [
 "I was charged twice for my monthly subscription", "please refund the duplicate payment on my invoice",
 "my invoice shows the wrong billing amount", "how do I update the credit card used for payment",
 "the discount code was not applied to my invoice", "I want a refund for the annual plan I never used",
 "payment failed but my card was charged", "need a receipt for last month payment",
 "the app crashes when I open the settings page", "login button does nothing on the mobile app",
 "error 500 appears when I save my profile", "the export freezes and the page never loads",
 "dark mode breaks the layout of the dashboard", "the mobile app crashes after the latest update",
 "search returns an error instead of results", "upload stops at ninety percent and shows an error",
 "how do I reset my password", "where can I find my api key in the dashboard",
 "how to export my data as a csv file", "how can I invite a teammate to my workspace",
 "how do I change the language of the app", "how do I enable two factor login",
 "how to connect the dashboard to my database", "how can I delete my account and data",
]
topic = ["billing"] * 8 + ["bug"] * 8 + ["how-to"] * 8

embedder = make_pipeline(TfidfVectorizer(sublinear_tf=True),
                         TruncatedSVD(n_components=8, random_state=0),
                         Normalizer())
E = embedder.fit_transform(docs).astype("float32")
print("embedding matrix:", E.shape, E.dtype, "row norms ~1:", np.allclose(np.linalg.norm(E, axis=1), 1, atol=1e-5))

def search(query, k=3):
    q = embedder.transform([query]).astype("float32")
    scores = (E @ q.T).ravel()                    # cosine, because rows are unit length
    top = np.argsort(-scores)[:k]
    return [(int(i), topic[i], round(float(scores[i]), 3)) for i in top]

for query in ["refund my payment", "the app crashes", "how can I export data"]:
    print(f"{query!r:26s}", search(query))

q = embedder.transform(["refund my payment"]).astype("float32")[0]
by_ip = np.argsort(-(E @ q))
by_l2 = np.argsort(((E - q) ** 2).sum(1))
print("same ranking by inner product and by L2:", np.array_equal(by_ip, by_l2))

embedding matrix: (24, 8) float32 row norms ~1: True
'refund my payment'        [(1, 'billing', 0.841), (7, 'billing', 0.711), (5, 'billing', 0.659)]
'the app crashes'          [(13, 'bug', 0.992), (8, 'bug', 0.957), (9, 'bug', 0.791)]
'how can I export data'    [(23, 'how-to', 0.973), (18, 'how-to', 0.962), (19, 'how-to', 0.798)]
same ranking by inner product and by L2: True


In [ ]:
import numpy as np

class FlatIndex:
    """NumPy imitation of faiss.IndexFlatIP / IndexFlatL2."""
    def __init__(self, d, metric="ip"):
        self.d, self.metric = d, metric
        self.x = np.zeros((0, d), dtype=np.float32)
    @property
    def ntotal(self):
        return len(self.x)
    def add(self, x):
        assert x.dtype == np.float32 and x.ndim == 2 and x.shape[1] == self.d, "need float32 (n, d)"
        self.x = np.vstack([self.x, x])
    def search(self, q, k):
        assert q.dtype == np.float32 and q.shape[1] == self.d
        if self.metric == "ip":
            scores = q @ self.x.T
            order = np.argsort(-scores, axis=1)[:, :k]
        else:
            scores = (q ** 2).sum(1)[:, None] - 2 * q @ self.x.T + (self.x ** 2).sum(1)[None, :]
            order = np.argsort(scores, axis=1)[:, :k]
        D = np.take_along_axis(scores, order, 1)
        I = order.copy()
        if k > self.ntotal:                                   # FAISS pads with -1
            pad = k - self.ntotal
            D = np.hstack([D, np.full((len(q), pad), np.nan, dtype=np.float32)])
            I = np.hstack([I, np.full((len(q), pad), -1)])
        return D.astype(np.float32), I

rng = np.random.default_rng(0)
x = rng.normal(size=(100, 16)).astype(np.float32)
x /= np.linalg.norm(x, axis=1, keepdims=True)
q = x[:2] + 0.05 * rng.normal(size=(2, 16)).astype(np.float32)
q /= np.linalg.norm(q, axis=1, keepdims=True)

ip, l2 = FlatIndex(16, "ip"), FlatIndex(16, "l2")
ip.add(x); l2.add(x)
D, I = ip.search(q, 3)
print("positions:", I.tolist())
print("scores   :", np.round(D.astype(float), 3).tolist())
print("L2 gives the same positions:", np.array_equal(I, l2.search(q, 3)[1]))

small = FlatIndex(16); small.add(x[:2])
print("k larger than ntotal:", small.search(q[:1], 4)[1].tolist())

try:
    ip.add(x.astype(np.float64))
except AssertionError as e:
    print("dtype guard:", e)

positions: [[0, 49, 60], [1, 62, 55]]
scores   : [[0.963, 0.537, 0.481], [0.977, 0.524, 0.519]]
L2 gives the same positions: True
k larger than ntotal: [[0, 1, -1, -1]]
dtype guard: need float32 (n, d)


In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs

def exact_topk(base, Q, k):
    d = (Q ** 2).sum(1)[:, None] - 2 * Q @ base.T + (base ** 2).sum(1)[None, :]
    idx = np.argpartition(d, k, axis=1)[:, :k]
    order = np.take_along_axis(d, idx, 1).argsort(1)
    return np.take_along_axis(idx, order, 1)

def ivf_curve(X, nlist=64, k=10, probes=(1, 2, 4, 8, 16, 32, 64)):
    base, Q = X[:20000], X[20000:20200]
    truth = exact_topk(base, Q, k)
    km = KMeans(n_clusters=nlist, n_init=1, random_state=0).fit(base)       # index.train
    lists = [np.where(km.labels_ == c)[0] for c in range(nlist)]               # index.add
    cent = km.cluster_centers_.astype(np.float32)
    out = []
    for nprobe in probes:
        rec, scanned = 0.0, 0
        for q, t in zip(Q, truth):
            probe = np.argsort(((cent - q) ** 2).sum(1))[:nprobe]
            cand = np.concatenate([lists[c] for c in probe])
            best = cand[np.argsort(((base[cand] - q) ** 2).sum(1))[:k]]
            rec += len(set(best) & set(t)) / k
            scanned += len(cand)
        out.append((nprobe, rec / len(Q), scanned / len(Q) / len(base)))
    return out

blobs, _ = make_blobs(n_samples=20200, n_features=32, centers=40, cluster_std=6.0, random_state=0)
noise = np.random.default_rng(0).normal(size=(20200, 32))
print("nprobe | clustered recall, scanned | gaussian recall, scanned")
for (p, r1, s1), (_, r2, s2) in zip(ivf_curve(blobs.astype(np.float32)), ivf_curve(noise.astype(np.float32))):
    print(f"{p:6d} | {r1:15.3f} {s1:9.3f} | {r2:15.3f} {s2:9.3f}")

nprobe | clustered recall, scanned | gaussian recall, scanned
     1 |           0.788     0.019 |           0.179     0.016
     2 |           0.964     0.033 |           0.285     0.031
     4 |           0.992     0.065 |           0.429     0.063
     8 |           0.997     0.129 |           0.620     0.125
    16 |           0.999     0.258 |           0.793     0.250
    32 |           1.000     0.510 |           0.942     0.501
    64 |           1.000     1.000 |           1.000     1.000


In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline

class Collection:
    def __init__(self, embedder):
        self.embedder, self.ids, self.docs, self.meta, self.E = embedder, [], [], [], None
    def add(self, ids, documents, metadatas):
        dup = set(ids) & set(self.ids)
        if dup:
            raise ValueError(f"duplicate ids: {sorted(dup)}")       # Chroma add() rejects existing ids
        self.ids += ids; self.docs += documents; self.meta += metadatas
        self.E = self.embedder.transform(self.docs)
    def query(self, query_texts, n_results=3, where=None):
        q = self.embedder.transform(query_texts)
        keep = [i for i, m in enumerate(self.meta) if not where or all(m.get(k) == v for k, v in where.items())]
        out = {"ids": [], "documents": [], "distances": []}
        for row in q:
            dist = 1 - self.E[keep] @ row                           # cosine distance, smaller is closer
            top = np.argsort(dist)[:n_results]
            out["ids"].append([self.ids[keep[j]] for j in top])
            out["documents"].append([self.docs[keep[j]] for j in top])
            out["distances"].append([round(float(dist[j]), 3) for j in top])
        return out
    def count(self):
        return len(self.ids)

docs = ["refund the duplicate payment", "charged twice this month", "app crashes on the settings page",
        "mobile login button does nothing", "how do I reset my password", "where is my api key",
        "mobile app crashes after update", "web dashboard layout is broken"]
meta = [{"label": "billing", "platform": "web"}, {"label": "billing", "platform": "web"},
        {"label": "bug", "platform": "mobile"}, {"label": "bug", "platform": "mobile"},
        {"label": "how-to", "platform": "web"}, {"label": "how-to", "platform": "web"},
        {"label": "bug", "platform": "mobile"}, {"label": "bug", "platform": "web"}]
embedder = make_pipeline(TfidfVectorizer(stop_words="english"), TruncatedSVD(n_components=5, random_state=0), Normalizer())
embedder.fit(docs)

col = Collection(embedder)
col.add([f"t{i}" for i in range(8)], docs, meta)
print("count:", col.count())
res = col.query(["the app keeps crashing"], n_results=2)
print(res["ids"], res["distances"])
res = col.query(["the app keeps crashing"], n_results=2, where={"platform": "web"})
print("filtered to web:", res["ids"], res["documents"])
try:
    col.add(["t0"], ["again"], [{"label": "bug", "platform": "web"}])
except ValueError as e:
    print("add twice:", e)

count: 8
[['t2', 't6']] [[0.007, 0.04]]
filtered to web: [['t7', 't4']] [['web dashboard layout is broken', 'how do I reset my password']]
add twice: duplicate ids: ['t0']


## 21.9 LLM SDK Calls: Clients, Retries, Streaming and Structured Output

In [ ]:
import random
from dataclasses import dataclass

class APIStatusError(Exception):
    def __init__(self, status_code, retry_after=None):
        super().__init__(f"HTTP {status_code}")
        self.status_code, self.retry_after = status_code, retry_after
class RateLimitError(APIStatusError): pass
class InternalServerError(APIStatusError): pass
class BadRequestError(APIStatusError): pass
class APITimeoutError(Exception): pass

@dataclass
class Response:
    text: str
    input_tokens: int
    output_tokens: int

class FakeLLM:
    """Plays back a script of outcomes: an exception instance to raise, or a string to return."""
    def __init__(self, script):
        self.script, self.calls = list(script), 0
    def complete(self, prompt):
        self.calls += 1
        outcome = self.script.pop(0)
        if isinstance(outcome, Exception):
            raise outcome
        return Response(outcome, input_tokens=len(prompt.split()), output_tokens=len(outcome.split()))

RETRYABLE = (RateLimitError, InternalServerError, APITimeoutError)

def call_with_retries(fn, max_retries=4, base=0.5, cap=8.0, sleep=None, rng=None):
    for attempt in range(max_retries + 1):
        try:
            return fn()
        except RETRYABLE as e:
            if attempt == max_retries:
                raise
            hinted = getattr(e, "retry_after", None)
            delay = hinted if hinted is not None else rng.uniform(0, min(cap, base * 2 ** attempt))
            sleep(delay)
        # BadRequestError is not in RETRYABLE, so it propagates immediately

def run(label, script):
    waits, llm = [], FakeLLM(script)
    try:
        r = call_with_retries(lambda: llm.complete("label this ticket please"),
                              sleep=waits.append, rng=random.Random(0))
        outcome = f"ok: {r.text!r}"
    except Exception as e:
        outcome = f"gave up: {type(e).__name__} ({e})"
    print(f"{label}: calls={llm.calls} waits={[round(w, 2) for w in waits]} -> {outcome}")

run("A", [RateLimitError(429, retry_after=2.0), InternalServerError(503), "billing"])
run("B", [BadRequestError(400), "never reached"])
run("C", [InternalServerError(503)] * 6)

A: calls=3 waits=[2.0, 0.84] -> ok: 'billing'
B: calls=1 waits=[] -> gave up: BadRequestError (HTTP 400)
C: calls=5 waits=[0.42, 0.76, 0.84, 1.04] -> gave up: InternalServerError (HTTP 503)


In [ ]:
import time

class FakeStreamingLLM:
    def __init__(self, text, fail_after=None):
        self.text, self.fail_after = text, fail_after
    def stream(self, prompt):
        words = self.text.split()
        for i, w in enumerate(words):
            if self.fail_after is not None and i == self.fail_after:
                raise ConnectionError("connection dropped mid-stream")
            yield w + (" " if i < len(words) - 1 else "")

def consume(stream_fn, on_piece=None):
    parts = []
    try:
        for piece in stream_fn():
            parts.append(piece)
            if on_piece:
                on_piece(piece)
        return "".join(parts), True
    except ConnectionError:
        return "".join(parts), False          # partial text plus a flag

reply = "Please reset your password using the link on the sign in page"
shown = []
text, ok = consume(lambda: FakeStreamingLLM(reply).stream("q"), on_piece=shown.append)
print("pieces:", len(shown), "| first piece:", repr(shown[0]), "| complete:", ok)
print("joined equals original:", text == reply)

text, ok = consume(lambda: FakeStreamingLLM(reply, fail_after=4).stream("q"))
print("failed stream ->", repr(text), "| complete:", ok)

def stream_with_fallback(prompt, attempts):
    for llm in attempts:                       # restart from scratch on a new attempt
        text, ok = consume(lambda: llm.stream(prompt))
        if ok:
            return text
    return None
print("recovered:", stream_with_fallback("q", [FakeStreamingLLM(reply, fail_after=2), FakeStreamingLLM(reply)]) == reply)

pieces: 12 | first piece: 'Please ' | complete: True
joined equals original: True
failed stream -> 'Please reset your password ' | complete: False
recovered: True


In [ ]:
import json, re
from dataclasses import dataclass

LABELS = {"billing", "bug", "how-to"}

class ValidationFailure(Exception):
    pass

@dataclass
class TicketLabel:
    label: str
    urgency: int

def parse_label(raw):
    text = raw.strip()
    fenced = re.match(r"^```(?:json)?\s*(.*?)\s*```$", text, re.S)       # models like to add code fences
    if fenced:
        text = fenced.group(1)
    try:
        data = json.loads(text)
    except json.JSONDecodeError as e:
        raise ValidationFailure(f"not valid JSON: {e.msg}")
    if not isinstance(data, dict):
        raise ValidationFailure("expected a JSON object")
    missing = {"label", "urgency"} - data.keys()
    if missing:
        raise ValidationFailure(f"missing keys: {sorted(missing)}")
    if data["label"] not in LABELS:
        raise ValidationFailure(f"label must be one of {sorted(LABELS)}, got {data['label']!r}")
    if not isinstance(data["urgency"], int) or isinstance(data["urgency"], bool) or not 1 <= data["urgency"] <= 5:
        raise ValidationFailure("urgency must be an integer from 1 to 5")
    return TicketLabel(data["label"], data["urgency"])

class ScriptedModel:
    def __init__(self, replies):
        self.replies, self.prompts = list(replies), []
    def complete(self, prompt):
        self.prompts.append(prompt)
        return self.replies.pop(0)

def classify(model, ticket, max_attempts=3):
    prompt = f"Return JSON with label and urgency for: {ticket}"
    for attempt in range(1, max_attempts + 1):
        raw = model.complete(prompt)
        try:
            return parse_label(raw), attempt
        except ValidationFailure as e:
            prompt = f"{prompt}\nYour last reply was rejected: {e}. Return only valid JSON."
    raise ValidationFailure("gave up after repairs")

replies = [
    '{"label": "billing", "urgency": 4,}',                     # trailing comma
    '```json\n{"label": "refund", "urgency": 4}\n```',          # fenced, bad label
    '{"label": "billing", "urgency": "high"}',                  # wrong type
    '{"label": "billing", "urgency": 4}',
]
m = ScriptedModel(replies)
try:
    result, attempts = classify(m, "charged twice", max_attempts=3)
except ValidationFailure as e:
    print("3 attempts failed:", e)
m = ScriptedModel(replies[1:])
result, attempts = classify(m, "charged twice", max_attempts=3)
print("result:", result, "after", attempts, "attempts")
print("last prompt tail:", m.prompts[-1].splitlines()[-1])
print("fenced valid reply parses:", parse_label('```json\n{"label": "bug", "urgency": 2}\n```'))

3 attempts failed: gave up after repairs
result: TicketLabel(label='billing', urgency=4) after 3 attempts
last prompt tail: Your last reply was rejected: urgency must be an integer from 1 to 5. Return only valid JSON.
fenced valid reply parses: TicketLabel(label='bug', urgency=2)


In [ ]:
import json

TICKETS = {7: {"status": "open", "plan": "pro"}, 9: {"status": "closed", "plan": "free"}}
def lookup_ticket(ticket_id):
    return TICKETS.get(ticket_id, {"error": "not found"})
TOOLS = {"lookup_ticket": lookup_ticket}

class ScriptedToolModel:
    """Turn 1 asks for a tool; turn 2 answers using the tool result."""
    def __init__(self):
        self.turn = 0
    def respond(self, messages):
        self.turn += 1
        if self.turn == 1:
            return {"type": "tool_call", "name": "lookup_ticket", "arguments": {"ticket_id": 7}}
        last = json.loads(messages[-1]["content"])
        return {"type": "text", "text": f"Ticket 7 is {last['status']} on the {last['plan']} plan."}

def agent_loop(model, question, max_turns=5):
    messages = [{"role": "user", "content": question}]
    for _ in range(max_turns):                                  # hard cap against endless loops
        out = model.respond(messages)
        if out["type"] == "text":
            return out["text"], messages
        fn = TOOLS.get(out["name"])
        result = fn(**out["arguments"]) if fn else {"error": "unknown tool"}
        messages.append({"role": "assistant", "content": f"call {out['name']}"})
        messages.append({"role": "tool", "content": json.dumps(result)})
    return "stopped: too many tool turns", messages

answer, msgs = agent_loop(ScriptedToolModel(), "What is the state of ticket 7?")
print(answer)
print("roles:", [m["role"] for m in msgs])

class TokenBucket:
    def __init__(self, rate_per_s, capacity):
        self.rate, self.capacity, self.tokens, self.t = rate_per_s, capacity, capacity, 0.0
    def allow(self, now, cost=1):
        self.tokens = min(self.capacity, self.tokens + (now - self.t) * self.rate)
        self.t = now
        if self.tokens >= cost:
            self.tokens -= cost
            return True
        return False

bucket = TokenBucket(rate_per_s=2, capacity=3)
times = [0.0, 0.1, 0.2, 0.3, 0.4, 1.0, 1.1]
print("allowed:", [bucket.allow(t) for t in times])

def cost_usd(inp, out, price_in_per_m=3.0, price_out_per_m=15.0):      # illustrative prices only
    return inp / 1e6 * price_in_per_m + out / 1e6 * price_out_per_m
print("10000 calls x (800 in, 150 out):", round(10000 * cost_usd(800, 150), 2), "dollars at the invented prices")

Ticket 7 is open on the pro plan.
roles: ['user', 'assistant', 'tool']
allowed: [True, True, True, False, False, True, True]
10000 calls x (800 in, 150 out): 46.5 dollars at the invented prices


## 21.10 Orchestration Awareness and Evaluation Harnesses

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline

class Runnable:
    def __init__(self, fn, name="step"):
        self.fn, self.name = fn, name
    def invoke(self, x):
        return self.fn(x)
    def batch(self, xs):
        return [self.invoke(x) for x in xs]
    def __or__(self, other):                       # a | b means: run a, then feed its output to b
        return Runnable(lambda x: other.invoke(self.invoke(x)), f"{self.name} | {other.name}")

docs = ["To reset your password open settings and click reset link",
        "Refunds are issued to the original card within five days",
        "Export your data from the dashboard menu as a csv file",
        "The mobile app crash was fixed in the latest update"]
embedder = make_pipeline(TfidfVectorizer(), TruncatedSVD(n_components=3, random_state=0), Normalizer()).fit(docs)
E = embedder.transform(docs)

def retrieve(question, k=2):
    scores = E @ embedder.transform([question])[0]
    return {"question": question, "docs": [docs[i] for i in np.argsort(-scores)[:k]]}

def build_prompt(d):
    context = "\n".join(f"- {t}" for t in d["docs"])
    return f"Answer from the context only.\nContext:\n{context}\nQuestion: {d['question']}"

def fake_model(prompt):
    lines = [l[2:] for l in prompt.splitlines() if l.startswith("- ")]
    return {"content": lines[0] if lines else "unknown", "prompt_chars": len(prompt)}   # echoes the best chunk

parse = Runnable(lambda msg: msg["content"].strip(), "parse")
chain = Runnable(retrieve, "retrieve") | Runnable(build_prompt, "prompt") | Runnable(fake_model, "model") | parse

print("chain:", chain.name)
print(chain.invoke("how do I reset my password"))
print(chain.batch(["where do refunds go", "export csv data"]))

prompt_text = (Runnable(retrieve) | Runnable(build_prompt)).invoke("where do refunds go")
print(prompt_text)

chain: retrieve | prompt | model | parse
To reset your password open settings and click reset link
['Refunds are issued to the original card within five days', 'Export your data from the dashboard menu as a csv file']
Answer from the context only.
Context:
- Refunds are issued to the original card within five days
- The mobile app crash was fixed in the latest update
Question: where do refunds go


In [ ]:
import numpy as np

def recall_at_k(ranked, relevant, k):
    return len(set(ranked[:k]) & set(relevant)) / len(relevant)

def precision_at_k(ranked, relevant, k):
    return len(set(ranked[:k]) & set(relevant)) / k

def mrr(ranked, relevant):
    for rank, item in enumerate(ranked, start=1):
        if item in relevant:
            return 1 / rank
    return 0.0

def ndcg_at_k(ranked, relevant, k):
    dcg = sum(1 / np.log2(r + 1) for r, item in enumerate(ranked[:k], start=1) if item in relevant)
    ideal = sum(1 / np.log2(r + 1) for r in range(1, min(len(relevant), k) + 1))
    return dcg / ideal

ranked, relevant = [7, 2, 9, 5, 1], {2, 5}
print("recall@3   :", recall_at_k(ranked, relevant, 3))
print("recall@5   :", recall_at_k(ranked, relevant, 5))
print("precision@3:", round(precision_at_k(ranked, relevant, 3), 4))
print("MRR        :", mrr(ranked, relevant))
print("nDCG@5     :", round(ndcg_at_k(ranked, relevant, 5), 4))

def token_f1(pred, gold):
    p, g = pred.lower().split(), gold.lower().split()
    common = sum(min(p.count(w), g.count(w)) for w in set(p))
    if common == 0:
        return 0.0
    prec, rec = common / len(p), common / len(g)
    return 2 * prec * rec / (prec + rec)
print("exact match:", "reset link" == "reset the link", "| token F1:", round(token_f1("reset link", "reset the link"), 3))

recall@3   : 0.5
recall@5   : 1.0
precision@3: 0.3333
MRR        : 0.5
nDCG@5     : 0.6509
exact match: False | token F1: 0.8


In [ ]:
import random
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix

rng = random.Random(0)
pools = {
    "billing": ["refund", "charged", "invoice", "payment", "card", "receipt", "subscription", "price", "billing", "discount"],
    "bug": ["crash", "error", "broken", "freezes", "fails", "slow", "bug", "blank", "timeout", "stuck"],
    "how-to": ["how", "export", "reset", "change", "find", "invite", "connect", "setup", "where", "enable"],
}
filler = ["the", "my", "app", "page", "account", "please", "when", "on", "I", "can", "with", "after", "today", "again", "dashboard"]
labels = list(pools)

def make(n):
    X, y = [], []
    for i in range(n):
        lab = labels[i % 3]
        words = rng.sample(pools[lab], rng.choice([1, 2]))
        if rng.random() < 0.6:                                    # confusing words from another class
            other = rng.choice([l for l in labels if l != lab])
            words += rng.sample(pools[other], rng.choice([1, 2]))
        words += rng.sample(filler, 3)
        rng.shuffle(words)
        X.append(" ".join(words)); y.append(lab)
    return X, np.array(y)

Xtr, ytr = make(300)
Xte, yte = make(150)

model = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
pred_model = model.predict(Xte)

def rule(text):                                                   # the baseline to beat
    w = set(text.split())
    if w & {"refund", "charged", "invoice", "payment"}: return "billing"
    if w & {"crash", "error", "broken", "freezes"}: return "bug"
    return "how-to"
pred_rule = np.array([rule(t) for t in Xte])

ok_model, ok_rule = (pred_model == yte).astype(int), (pred_rule == yte).astype(int)
print("accuracy model:", round(ok_model.mean(), 3), " rule baseline:", round(ok_rule.mean(), 3))
print("model wins / baseline wins:", int(((ok_model == 1) & (ok_rule == 0)).sum()), "/", int(((ok_model == 0) & (ok_rule == 1)).sum()))

boot = np.random.default_rng(0)
diffs = []
for _ in range(2000):
    idx = boot.integers(0, len(yte), len(yte))
    diffs.append(ok_model[idx].mean() - ok_rule[idx].mean())
lo, hi = np.percentile(diffs, [2.5, 97.5])
print(f"accuracy gain 95% CI: [{lo:.3f}, {hi:.3f}]")
print("confusion matrix (rows true, columns predicted), order", labels)
print(confusion_matrix(yte, pred_model, labels=labels))

accuracy model: 0.68  rule baseline: 0.56
model wins / baseline wins: 37 / 19
accuracy gain 95% CI: [0.027, 0.213]
confusion matrix (rows true, columns predicted), order ['billing', 'bug', 'how-to']
[[31 13  6]
 [ 7 38  5]
 [ 9  8 33]]


In [ ]:
import numpy as np
from sklearn.metrics import cohen_kappa_score

human = np.array([1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1])
judge = np.array([1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1])

agree = (human == judge).mean()
print("raw agreement:", agree)
print("kappa:", round(cohen_kappa_score(human, judge), 3))
print("judge says 'good' on", int(judge.sum()), "of", len(judge), "| humans say 'good' on", int(human.sum()))
fp = int(((judge == 1) & (human == 0)).sum())
print("judge approves", fp, "replies that humans rejected")

always_yes = np.ones_like(human)
print("an always-yes judge: agreement", (human == always_yes).mean(), "kappa", round(cohen_kappa_score(human, always_yes), 3))

raw agreement: 0.9
kappa: 0.765
judge says 'good' on 15 of 20 | humans say 'good' on 13
judge approves 2 replies that humans rejected
an always-yes judge: agreement 0.65 kappa 0.0


## 21.11 Recipes: How Do I ...?

Each recipe is a small, real task. Torch snippets are marked not run here; the ones with output are runnable stand-ins you can execute.

In [ ]:
import numpy as np

y = np.array([0] * 90 + [1] * 8 + [2] * 2)           # 90 billing, 8 bug, 2 outage
counts = np.bincount(y)
weights = len(y) / (len(counts) * counts)
print("counts :", counts)
print("weights:", np.round(weights, 3))

def weighted_ce(logits, y, w):
    z = logits - logits.max(1, keepdims=True)
    logp = z - np.log(np.exp(z).sum(1, keepdims=True))
    per = -logp[np.arange(len(y)), y]
    return (w[y] * per).sum() / w[y].sum()           # PyTorch normalises by the sum of weights too

logits = np.zeros((len(y), 3))                       # an uninformed model
print("plain loss   :", round(float(-np.log(1 / 3)), 4))
print("weighted loss:", round(float(weighted_ce(logits, y, weights)), 4))
logits[:, 0] = 3.0                                   # a model that always says class 0
print("always-0 plain accuracy:", (logits.argmax(1) == y).mean(), "| weighted loss:", round(float(weighted_ce(logits, y, weights)), 3))

counts : [90  8  2]
weights: [ 0.37   4.167 16.667]
plain loss   : 1.0986
weighted loss: 1.0986
always-0 plain accuracy: 0.9 | weighted loss: 2.095


In [ ]:
import numpy as np

lengths = np.array([5, 40, 7, 38, 6, 41, 9, 37, 8, 39])
order = np.argsort(lengths)                          # short with short, long with long
batches = [order[i:i + 5] for i in range(0, len(order), 5)]
padded_sorted = sum(lengths[b].max() * len(b) for b in batches)
unsorted = [np.arange(10)[i:i + 5] for i in range(0, 10, 5)]
padded_unsorted = sum(lengths[b].max() * len(b) for b in unsorted)
print("padded cells, unsorted batches:", padded_unsorted, "| sorted batches:", padded_sorted)

preds_sorted = lengths[order] * 2                    # pretend model output, in sorted order
restored = np.empty_like(preds_sorted)
restored[order] = preds_sorted                       # undo the sort
print("original order restored:", np.array_equal(restored, lengths * 2))

padded cells, unsorted batches: 405 | sorted batches: 250
original order restored: True


In [ ]:
def chunk(words, size, overlap):
    assert 0 <= overlap < size
    step = size - overlap
    return [words[i:i + size] for i in range(0, max(1, len(words) - overlap), step)]

text = "one two three four five six seven eight nine ten eleven twelve".split()
for c in chunk(text, size=5, overlap=2):
    print(c)
print("chunks for 1000 words, size 200, overlap 50:", len(chunk(list(range(1000)), 200, 50)))

['one', 'two', 'three', 'four', 'five']
['four', 'five', 'six', 'seven', 'eight']
['seven', 'eight', 'nine', 'ten', 'eleven']
['ten', 'eleven', 'twelve']
chunks for 1000 words, size 200, overlap 50: 7


In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

tickets = ["charged twice this month", "I was charged twice this month", "app crashes on login",
           "the app crashes on login screen", "how do I export data"]
M = TfidfVectorizer(ngram_range=(1, 2)).fit_transform(tickets)
S = (M @ M.T).toarray()                       # TF-IDF rows are unit length, so this is cosine
np.fill_diagonal(S, 0)
i, j = np.where(np.triu(S) > 0.5)
for a, b in zip(i, j):
    print(f"{a} ~ {b}  cosine {S[a, b]:.2f}  {tickets[a]!r} / {tickets[b]!r}")

0 ~ 1  cosine 0.83  'charged twice this month' / 'I was charged twice this month'
2 ~ 3  cosine 0.73  'app crashes on login' / 'the app crashes on login screen'


In [ ]:
import hashlib, json

class CachedClient:
    def __init__(self, llm):
        self.llm, self.cache, self.hits = llm, {}, 0
    def complete(self, model, prompt, temperature=0):
        key = hashlib.sha256(json.dumps([model, prompt, temperature], sort_keys=True).encode()).hexdigest()
        if key in self.cache:
            self.hits += 1
            return self.cache[key]
        self.cache[key] = self.llm(prompt)
        return self.cache[key]

calls = []
def llm(prompt):
    calls.append(prompt)
    return prompt.upper()

c = CachedClient(llm)
for p in ["hello", "hello", "bye", "hello"]:
    c.complete("m1", p)
print("real calls:", len(calls), "| cache hits:", c.hits)
c.complete("m2", "hello")
print("different model -> new call, total real calls:", len(calls))

real calls: 2 | cache hits: 2
different model -> new call, total real calls: 3


In [ ]:
import asyncio

active, peak = 0, 0

async def fake_call(i):
    global active, peak
    active += 1
    peak = max(peak, active)
    await asyncio.sleep(0.001)
    active -= 1
    return i * 2

async def run_all(n, limit):
    sem = asyncio.Semaphore(limit)
    async def guarded(i):
        async with sem:
            return await fake_call(i)
    return await asyncio.gather(*(guarded(i) for i in range(n)))

results = asyncio.run(run_all(200, limit=8))
print("results in input order:", results[:5], "... count", len(results))
print("peak concurrent calls:", peak)

results in input order: [0, 2, 4, 6, 8] ... count 200
peak concurrent calls: 8


In [ ]:
import numpy as np

def noisy_mean(seed):
    rng = np.random.default_rng(seed)
    return float(rng.normal(size=1000).mean())

print(noisy_mean(7) == noisy_mean(7), noisy_mean(7) == noisy_mean(8))
child1, child2 = np.random.default_rng(1), np.random.default_rng(2)
print("independent streams differ:", child1.random() != child2.random())

True False
independent streams differ: True


In [ ]:
def fit_history(system, turns, budget, count=lambda s: len(s.split())):
    used = count(system)
    kept = []
    for role, text in reversed(turns):            # walk from the newest turn backwards
        c = count(text)
        if used + c > budget:
            break
        kept.append((role, text)); used += c
    return list(reversed(kept)), used

turns = [("user", "hello there"), ("assistant", "hi how can I help you today"),
         ("user", "my invoice is wrong"), ("assistant", "which month is it from"), ("user", "last month")]
kept, used = fit_history("You are a billing assistant", turns, budget=16)
print([t[1] for t in kept], "| used:", used)

['my invoice is wrong', 'which month is it from', 'last month'] | used: 16


In [ ]:
import numpy as np

labels = np.array(["billing", "bug", "how-to", "other"])
logits = np.array([[2.0, 0.5, -1.0, 0.0], [0.1, 0.2, 3.0, 0.0]])
z = logits - logits.max(1, keepdims=True)
probs = np.exp(z) / np.exp(z).sum(1, keepdims=True)
top2 = np.argsort(-probs, axis=1)[:, :2]                    # torch.topk(probs, 2)
for row, idx in enumerate(top2):
    print([(str(labels[i]), round(float(probs[row, i]), 3)) for i in idx])

[('billing', 0.71), ('bug', 0.158)]
[('how-to', 0.858), ('bug', 0.052)]


## 21.12 Performance and Pitfalls

Twelve traps, each with the wrong code, the symptom and the fix. The numbers in this section are timings measured on the machine that produced this book; they vary on yours, so measure your own.

In [ ]:
import time
import numpy as np

rng = np.random.default_rng(0)
X = rng.normal(size=(2000, 64)).astype(np.float32)
W = rng.normal(size=(64, 32)).astype(np.float32)

t0 = time.perf_counter(); slow = np.stack([x @ W for x in X]); t_loop = time.perf_counter() - t0
t0 = time.perf_counter(); fast = X @ W; t_batch = time.perf_counter() - t0
print("same result:", np.allclose(slow, fast, atol=1e-4))
print("batched is more than 2x faster:", t_loop / t_batch > 2)

same result: True
batched is more than 2x faster: True


## 21.13 Common Error Messages and Their Fixes

Exact wording differs between library versions, so search for the distinctive phrase. The table lists the situation, the cause and the fix.

In [ ]:
import numpy as np

x = np.ones((32, 10)); W = np.ones((20, 5))
try:
    x @ W
except ValueError as e:
    print("numpy:", str(e).splitlines()[0][:70])
try:
    np.array([[1, 2, 3, 4, 5], [1, 2, 3]], dtype=float)
except ValueError as e:
    print("ragged:", str(e)[:80])
t = np.arange(12).reshape(3, 4).T
try:
    t.view().shape = (12,)
except AttributeError as e:
    print("reshape in place:", str(e)[:70])

numpy: matmul: Input operand 1 has a mismatch in its core dimension 0, with g
ragged: setting an array element with a sequence. The requested array has an inhomogeneo
reshape in place: Incompatible shape for in-place modification. Use `.reshape()` to make


## 21.14 Cross-Library Integration

Real projects pass data between libraries. Most integration bugs are one of three kinds: **dtype** (float64 versus float32, int32 versus int64), **memory sharing** (a view changes the original), and **location** (CPU versus GPU, or a tracked tensor versus plain NumPy). The table is the handoff cheat sheet.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

df = pd.DataFrame({
    "text": ["refund my charge", "charged twice", "invoice is wrong", "refund the payment",
             "app crashes", "error on save", "page is broken", "login fails with error"],
    "label": ["billing", "billing", "billing", "billing", "bug", "bug", "bug", "bug"],
})
print("text dtype:", df["text"].dtype, "| label dtype:", df["label"].dtype)

cat = pd.Categorical(df["label"])
y = cat.codes.astype(np.int64)                        # class ids: int64 is what PyTorch's CrossEntropy wants
X = TfidfVectorizer().fit_transform(df["text"]).toarray().astype(np.float32)
X = np.ascontiguousarray(X)
print("X:", X.shape, X.dtype, "| y:", y.dtype, "| classes:", list(cat.categories))

view = X[:2]
print("slice shares memory (torch.from_numpy would too):", np.shares_memory(X, view))

clf = LogisticRegression(max_iter=200).fit(X, y)
pred = clf.predict(X)
print("accuracy:", accuracy_score(y, pred))

out = df.assign(pred=cat.categories[pred], correct=(pred == y))     # a NEW frame (pandas 3 copy-on-write)
subset = out[out["label"] == "bug"]
subset.loc[subset.index[0], "pred"] = "changed"                      # touches only the subset
print("original untouched:", out.loc[4, "pred"] != "changed", "| rows:", len(out), "columns:", list(out.columns))

text dtype: str | label dtype: str
X: (8, 20) float32 | y: int64 | classes: ['billing', 'bug']
slice shares memory (torch.from_numpy would too): True
accuracy: 1.0
original untouched: True | rows: 8 columns: ['text', 'label', 'pred', 'correct']


In [ ]:
import json, threading, time
import httpx
from http.server import BaseHTTPRequestHandler, HTTPServer

state = {"calls": 0}

class Handler(BaseHTTPRequestHandler):
    def do_POST(self):
        state["calls"] += 1
        body = json.loads(self.rfile.read(int(self.headers["Content-Length"])))
        if state["calls"] == 1:
            self.send_response(429); self.send_header("Retry-After", "0.05"); self.end_headers(); return
        reply = json.dumps({"text": body["prompt"].upper(), "usage": {"input_tokens": 3, "output_tokens": 3}}).encode()
        self.send_response(200); self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(reply))); self.end_headers(); self.wfile.write(reply)
    def log_message(self, *a): pass

srv = HTTPServer(("127.0.0.1", 0), Handler)
threading.Thread(target=srv.serve_forever, daemon=True).start()
url = f"http://127.0.0.1:{srv.server_address[1]}/v1/complete"

def post_with_retry(client, url, payload, max_retries=3):
    for attempt in range(max_retries + 1):
        r = client.post(url, json=payload, headers={"Authorization": "Bearer demo-key"})
        if r.status_code == 429 and attempt < max_retries:
            time.sleep(float(r.headers.get("Retry-After", 1)))
            continue
        r.raise_for_status()                      # 4xx/5xx other than a retried 429 raise here
        return r.json()

with httpx.Client(timeout=5.0) as client:
    data = post_with_retry(client, url, {"prompt": "hello llm"})
print(data["text"], "| usage:", data["usage"], "| server saw", state["calls"], "requests")
srv.shutdown()

HELLO LLM | usage: {'input_tokens': 3, 'output_tokens': 3} | server saw 2 requests


## 21.15 How to Solve Any Deep Learning and GenAI Libraries Problem

In [ ]:
import numpy as np

def make_data():
    rng = np.random.default_rng(0)
    X = rng.normal(size=(8, 4))
    y = np.array([0, 1, 0, 1, 1, 0, 1, 0])
    return X, y, rng

def train(sign, steps=300, lr=0.5):
    X, y, rng = make_data()
    W1, b1 = rng.normal(scale=0.5, size=(4, 16)), np.zeros(16)
    W2, b2 = rng.normal(scale=0.5, size=(16, 2)), np.zeros(2)
    losses = []
    for _ in range(steps):
        h = np.maximum(X @ W1 + b1, 0)
        z = h @ W2 + b2
        z -= z.max(1, keepdims=True)
        p = np.exp(z); p /= p.sum(1, keepdims=True)
        losses.append(-np.log(p[np.arange(8), y]).mean())
        g = p.copy(); g[np.arange(8), y] -= 1; g /= 8
        gW2, gb2 = h.T @ g, g.sum(0)
        gh = (g @ W2.T) * (h > 0)
        gW1, gb1 = X.T @ gh, gh.sum(0)
        W1 -= sign * lr * gW1; b1 -= sign * lr * gb1
        W2 -= sign * lr * gW2; b2 -= sign * lr * gb2
    return losses

good, bad = train(+1), train(-1)
print("correct update: loss", round(good[0], 3), "->", f"{good[-1]:.4f}")
print("sign bug      : loss", round(bad[0], 3), "->", round(bad[-1], 3))

correct update: loss 0.715 -> 0.0029
sign bug      : loss 0.715 -> inf


In [ ]:
import numpy as np

def run(lr, clip=None, steps=400):
    x = 3.0                                              # minimise 50 * x^2, gradient 100 * x
    for step in range(1, steps + 1):
        g = 100 * x
        if clip is not None and abs(g) > clip:
            g = clip * np.sign(g)                        # clip the gradient
        x = x - lr * g
        if not np.isfinite(50 * x * x):
            return f"loss not finite at step {step}"
    return f"finished, x = {x:.4f}"

with np.errstate(all="ignore"):
    print("lr 0.05           :", run(0.05))
    print("lr 0.05 + clip 5  :", run(0.05, clip=5.0))
    print("lr 0.005          :", run(0.005))

lr 0.05           : loss not finite at step 254
lr 0.05 + clip 5  : finished, x = 0.0000
lr 0.005          : finished, x = 0.0000


In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import make_pipeline

docs = [
 "I was charged twice for my monthly subscription", "please refund the duplicate payment on my invoice",
 "my invoice shows the wrong billing amount", "how do I update the credit card used for payment",
 "the discount code was not applied to my invoice", "I want a refund for the annual plan I never used",
 "payment failed but my card was charged", "need a receipt for last month payment",
 "the app crashes when I open the settings page", "login button does nothing on the mobile app",
 "error 500 appears when I save my profile", "the export freezes and the page never loads",
 "dark mode breaks the layout of the dashboard", "the mobile app crashes after the latest update",
 "search returns an error instead of results", "upload stops at ninety percent and shows an error",
 "how do I reset my password", "where can I find my api key in the dashboard",
 "how to export my data as a csv file", "how can I invite a teammate to my workspace",
 "how do I change the language of the app", "how do I enable two factor login",
 "how to connect the dashboard to my database", "how can I delete my account and data",
]
topic = np.array(["billing"] * 8 + ["bug"] * 8 + ["how-to"] * 8)
queries = [("my card was billed two times", "billing"), ("give me my money back", "billing"),
           ("receipt for payment", "billing"), ("the mobile app keeps crashing", "bug"),
           ("error when saving", "bug"), ("page freezes", "bug"),
           ("how can I export my data", "how-to"), ("where is the api key", "how-to"),
           ("how do I invite someone", "how-to")]

def precision_at_3(embed):
    E = embed.fit_transform(docs)
    hits = 0
    for q, t in queries:
        s = E @ embed.transform([q])[0] if not hasattr(E, "toarray") else (E @ embed.transform([q]).T).toarray().ravel()
        top = np.argsort(-s)[:3]
        hits += (topic[top] == t).sum()
    return hits / (3 * len(queries))

variants = {
    "tf-idf words only": make_pipeline(TfidfVectorizer(), Normalizer()),
    "tf-idf + SVD 2 dims": make_pipeline(TfidfVectorizer(), TruncatedSVD(2, random_state=0), Normalizer()),
    "tf-idf + SVD 8 dims": make_pipeline(TfidfVectorizer(), TruncatedSVD(8, random_state=0), Normalizer()),
}
for name, v in variants.items():
    print(f"{name:22s} precision@3 = {precision_at_3(v):.3f}")

tf-idf words only      precision@3 = 0.741
tf-idf + SVD 2 dims    precision@3 = 0.741
tf-idf + SVD 8 dims    precision@3 = 0.889


In [ ]:
import time
import numpy as np

class Timer:
    def __init__(self): self.t = {}
    def stage(self, name):
        timer = self
        class Ctx:
            def __enter__(s): s.t0 = time.perf_counter()
            def __exit__(s, *a): timer.t[name] = timer.t.get(name, 0) + time.perf_counter() - s.t0
        return Ctx()

texts = ["support ticket number %d about billing and refunds" % i for i in range(3000)]
W = np.random.default_rng(0).normal(size=(64, 64)).astype(np.float32)
tm = Timer()
for start in range(0, 3000, 100):
    with tm.stage("data"):                         # python-level tokenising and hashing per item
        batch = np.array([[hash((w, j)) % 97 for j, w in enumerate(t.split())] + [0] * 56 for t in texts[start:start + 100]],
                         dtype=np.float32)[:, :64]
    with tm.stage("compute"):
        out = batch @ W
slowest = max(tm.t, key=tm.t.get)
print("stages measured:", sorted(tm.t))
print("the slower stage in this run is:", slowest)

stages measured: ['compute', 'data']
the slower stage in this run is: data


## 21.16 Going Deeper: Internals, Memory, Speed and What Interviewers Probe

In [ ]:
import numpy as np

def probs_from_logits(logits, temperature=1.0, top_k=None, top_p=None):
    z = np.asarray(logits, dtype=float) / temperature
    p = np.exp(z - z.max()); p /= p.sum()
    if top_k is not None:
        cutoff = np.sort(p)[-top_k]
        p = np.where(p >= cutoff, p, 0.0)
    if top_p is not None:
        order = np.argsort(-p)
        cum = np.cumsum(p[order])
        keep = order[: np.searchsorted(cum, top_p) + 1]      # smallest prefix reaching top_p
        mask = np.zeros_like(p); mask[keep] = 1
        p = p * mask
    return p / p.sum()

logits = [4.0, 3.0, 2.0, 1.0, 0.0]
for name, kw in [("T=1", {}), ("T=0.5 (sharper)", {"temperature": 0.5}), ("T=2 (flatter)", {"temperature": 2.0}),
                 ("top_k=2", {"top_k": 2}), ("top_p=0.9", {"top_p": 0.9})]:
    print(f"{name:16s}", np.round(probs_from_logits(logits, **kw), 3))

rng = np.random.default_rng(0)
draws = rng.choice(5, size=1000, p=probs_from_logits(logits, top_k=2))
print("top_k=2 draws only use tokens:", sorted(set(draws.tolist())))

T=1              [0.636 0.234 0.086 0.032 0.012]
T=0.5 (sharper)  [0.865 0.117 0.016 0.002 0.   ]
T=2 (flatter)    [0.429 0.26  0.158 0.096 0.058]
top_k=2          [0.731 0.269 0.    0.    0.   ]
top_p=0.9        [0.665 0.245 0.09  0.    0.   ]
top_k=2 draws only use tokens: [0, 1]


## Chapter recap

## Practice set

Graded from easy to expert. Code shown in a problem is not run by the book; answers were checked by running equivalent code.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/python/ch21.html